# 01 — Internal Data: Portfolio Tables + Research Documents

Loads the **internal** side of the demo into Unity Catalog Delta tables:

| Data | Tables | Consumed by |
|---|---|---|
| **Structured** | `accounts`, `portfolios`, `holdings`, `transactions` | AI/BI **Genie** space + Unity Catalog **SQL functions** (structured tool) |
| **Unstructured** | `research_documents` | **Vector Search** index (unstructured tool, notebook `02`) |

This is representative institutional portfolio data — 3 accounts, 3 portfolios,
15 holdings, 20 transactions, and 6 internal research notes (theses, a strategy
memo, and a risk assessment). It mirrors the dataset used by the other
`*_Agent_To_Bigdata` cookbooks so the demos are directly comparable.

In [0]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

## 1. Structured tables

In [0]:
%sql
CREATE OR REPLACE TABLE accounts (
    account_id   STRING,
    account_name STRING,
    account_type STRING,
    currency     STRING,
    balance      DECIMAL(18,2)
) COMMENT 'Top-level institutional accounts';

INSERT INTO accounts VALUES
    ('ACC001', 'Institutional Growth Fund', 'Institutional', 'USD',  50000000),
    ('ACC002', 'Tech Innovation Portfolio', 'Hedge Fund',    'USD',  25000000),
    ('ACC003', 'Global Macro Strategy',     'Pension Fund',  'USD', 100000000);

In [0]:
%sql
CREATE OR REPLACE TABLE portfolios (
    portfolio_id   STRING,
    portfolio_name STRING,
    account_id     STRING,
    strategy       STRING,
    risk_profile   STRING,
    aum            DECIMAL(18,2)
) COMMENT 'Portfolios belonging to accounts';

INSERT INTO portfolios VALUES
    ('PF001', 'US Large Cap Growth',      'ACC001', 'Growth',       'Moderate',   30000000),
    ('PF002', 'AI & Semiconductor Focus', 'ACC002', 'Sector Focus', 'Aggressive', 15000000),
    ('PF003', 'Diversified Tech Leaders', 'ACC003', 'Value Growth', 'Moderate',   50000000);

In [0]:
%sql
CREATE OR REPLACE TABLE holdings (
    holding_id     BIGINT GENERATED ALWAYS AS IDENTITY,
    portfolio_id   STRING,
    ticker         STRING,
    company_name   STRING,
    shares         DECIMAL(18,4),
    avg_cost       DECIMAL(18,2),
    current_price  DECIMAL(18,2),
    market_value   DECIMAL(18,2),
    unrealized_pnl DECIMAL(18,2),
    weight_pct     DECIMAL(8,2)
) COMMENT 'Current positions per portfolio';

INSERT INTO holdings
  (portfolio_id, ticker, company_name, shares, avg_cost, current_price, market_value, unrealized_pnl, weight_pct) VALUES
    ('PF001', 'AAPL',  'Apple Inc.',              15000,  142.50, 185.25, 2778750,  641250,  9.26),
    ('PF001', 'MSFT',  'Microsoft Corporation',    8000,  285.00, 415.50, 3324000, 1044000, 11.08),
    ('PF001', 'GOOGL', 'Alphabet Inc.',            5000,  125.00, 175.25,  876250,  251250,  2.92),
    ('PF001', 'AMZN',  'Amazon.com Inc.',          6000,  145.00, 225.75, 1354500,  484500,  4.52),
    ('PF001', 'META',  'Meta Platforms Inc.',      4500,  280.00, 585.00, 2632500, 1372500,  8.78),
    ('PF002', 'NVDA',  'NVIDIA Corporation',      12000,  450.00, 875.50,10506000, 5106000, 70.04),
    ('PF002', 'AMD',   'Advanced Micro Devices',   8000,   95.00, 145.25, 1162000,  402000,  7.75),
    ('PF002', 'AVGO',  'Broadcom Inc.',            1500,  850.00,1425.00, 2137500,  862500, 14.25),
    ('PF002', 'TSM',   'Taiwan Semiconductor',     3000,  110.00, 185.75,  557250,  227250,  3.72),
    ('PF002', 'PLTR',  'Palantir Technologies',   25000,   18.50,  65.25, 1631250, 1168750, 10.88),
    ('PF003', 'AAPL',  'Apple Inc.',              25000,  155.00, 185.25, 4631250,  756250,  9.26),
    ('PF003', 'MSFT',  'Microsoft Corporation',   15000,  310.00, 415.50, 6232500, 1582500, 12.47),
    ('PF003', 'NVDA',  'NVIDIA Corporation',       8000,  520.00, 875.50, 7004000, 2844000, 14.01),
    ('PF003', 'CRM',   'Salesforce Inc.',         10000,  215.00, 325.50, 3255000, 1105000,  6.51),
    ('PF003', 'ORCL',  'Oracle Corporation',      12000,   95.00, 175.25, 2103000,  963000,  4.21);

In [0]:
%sql
CREATE OR REPLACE TABLE transactions (
    transaction_id   BIGINT GENERATED ALWAYS AS IDENTITY,
    portfolio_id     STRING,
    ticker           STRING,
    transaction_type STRING,
    shares           DECIMAL(18,4),
    price            DECIMAL(18,2),
    amount           DECIMAL(18,2),
    fees             DECIMAL(18,2),
    transaction_date TIMESTAMP,
    notes            STRING
) COMMENT 'Trade blotter';

INSERT INTO transactions
  (portfolio_id, ticker, transaction_type, shares, price, amount, fees, transaction_date, notes) VALUES
    ('PF001', 'AAPL',  'BUY',       500,  178.50,  89250.00,  44.63, '2025-01-08T10:30:00', 'BUY order for AAPL'),
    ('PF001', 'MSFT',  'BUY',       200,  405.25,  81050.00,  40.53, '2025-01-12T14:15:00', 'BUY order for MSFT'),
    ('PF001', 'GOOGL', 'SELL',      300,  172.00,  51600.00,  25.80, '2025-01-15T09:45:00', 'SELL order for GOOGL'),
    ('PF001', 'META',  'BUY',       150,  575.00,  86250.00,  43.13, '2025-01-20T11:00:00', 'BUY order for META'),
    ('PF001', 'AAPL',  'DIVIDEND',15000,    0.25,   3750.00,   0.00, '2025-02-01T00:00:00', 'DIVIDEND for AAPL'),
    ('PF001', 'AMZN',  'BUY',       400,  220.00,  88000.00,  44.00, '2025-02-05T13:20:00', 'BUY order for AMZN'),
    ('PF001', 'MSFT',  'SELL',      100,  420.00,  42000.00,  21.00, '2025-02-10T15:30:00', 'SELL order for MSFT'),
    ('PF002', 'NVDA',  'BUY',      1000,  850.00, 850000.00, 425.00, '2025-01-06T09:30:00', 'BUY order for NVDA'),
    ('PF002', 'AMD',   'SELL',      500,  148.00,  74000.00,  37.00, '2025-01-10T10:00:00', 'SELL order for AMD'),
    ('PF002', 'PLTR',  'BUY',      2000,   62.00, 124000.00,  62.00, '2025-01-18T14:00:00', 'BUY order for PLTR'),
    ('PF002', 'AVGO',  'BUY',       100, 1400.00, 140000.00,  70.00, '2025-01-25T11:30:00', 'BUY order for AVGO'),
    ('PF002', 'NVDA',  'SELL',      200,  890.00, 178000.00,  89.00, '2025-02-03T09:15:00', 'SELL order for NVDA'),
    ('PF002', 'TSM',   'BUY',       500,  180.00,  90000.00,  45.00, '2025-02-08T10:45:00', 'BUY order for TSM'),
    ('PF002', 'AMD',   'BUY',       800,  140.00, 112000.00,  56.00, '2025-02-15T13:00:00', 'BUY order for AMD'),
    ('PF003', 'AAPL',  'BUY',      1500,  180.00, 270000.00, 135.00, '2025-01-07T10:00:00', 'BUY order for AAPL'),
    ('PF003', 'MSFT',  'BUY',       500,  400.00, 200000.00, 100.00, '2025-01-14T11:30:00', 'BUY order for MSFT'),
    ('PF003', 'NVDA',  'BUY',       300,  860.00, 258000.00, 129.00, '2025-01-22T09:00:00', 'BUY order for NVDA'),
    ('PF003', 'CRM',   'SELL',      200,  330.00,  66000.00,  33.00, '2025-01-28T14:45:00', 'SELL order for CRM'),
    ('PF003', 'ORCL',  'BUY',      1000,  170.00, 170000.00,  85.00, '2025-02-06T10:30:00', 'BUY order for ORCL'),
    ('PF003', 'MSFT',  'DIVIDEND',15000,    0.75,  11250.00,   0.00, '2025-02-14T00:00:00', 'DIVIDEND for MSFT');

In [0]:
%sql
SELECT 'accounts' AS table_name, COUNT(*) AS rows FROM accounts
UNION ALL SELECT 'portfolios',   COUNT(*) FROM portfolios
UNION ALL SELECT 'holdings',     COUNT(*) FROM holdings
UNION ALL SELECT 'transactions', COUNT(*) FROM transactions;

## 2. Unstructured: internal research documents

Six internal research notes. Notebook `02` builds a Vector Search index over the
`content` column so the agent can semantically search proprietary research.

In [0]:
%sql
CREATE OR REPLACE TABLE research_documents (
    doc_id   STRING,
    title    STRING,
    content  STRING,
    ticker   STRING,
    company  STRING,
    doc_type STRING,
    doc_date DATE
)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
COMMENT 'Internal research notes — vector-searchable proprietary content';

In [0]:
research_docs = [
    ("DOC001", "NVIDIA Q4 2024 Investment Thesis Update",
     """NVIDIA Q4 2024 Investment Thesis Update

NVIDIA remains our top pick in the semiconductor space. Key highlights:

1. Data Center Revenue: $18.4B (+409% YoY) driven by H100/H200 GPU demand for AI training
2. Blackwell Architecture: Next-gen B100/B200 GPUs launching Q2 2025 with 2.5x performance
3. Software Moat: CUDA ecosystem has 4M+ developers, creating significant switching costs
4. AI Inference Opportunity: $150B TAM by 2027 as enterprises deploy AI at scale

Risk Factors: China export restrictions, AMD competition, supply constraints
Price Target: $950 (25x FY26E EPS)
Rating: STRONG BUY""",
     "NVDA", "NVIDIA", "investment_thesis", "2024-12-15"),

    ("DOC002", "Apple Inc. Strategic Analysis - Services & AI Focus",
     """Apple Inc. Strategic Analysis - Services & AI Focus

Key Investment Points:

1. Services Segment ($96B ARR): Highest-margin business (70%+ gross margin)
   - App Store, Apple Music, iCloud, Apple TV+, Apple Pay
   - 1B+ paid subscriptions across ecosystem

2. Apple Intelligence (AI Strategy):
   - On-device AI processing preserving privacy
   - Partnership with OpenAI for ChatGPT integration
   - Siri 2.0 with LLM capabilities launching iOS 18.4

3. iPhone 16 Cycle:
   - AI features driving upgrade demand
   - Pro models with A18 Pro chip outperforming

Valuation: Trading at 28x FY25E P/E, premium justified by ecosystem strength
Price Target: $210""",
     "AAPL", "Apple", "strategic_analysis", "2024-12-12"),

    ("DOC003", "Microsoft Azure & AI Monetization Analysis",
     """Microsoft Azure & AI Monetization Analysis

Cloud & AI Revenue Breakdown:

1. Azure Growth: +29% YoY (Q1 FY25)
   - AI services contributing 12 percentage points to growth
   - 60K+ Azure AI customers (2x YoY)
   - OpenAI partnership generating $3B+ annual revenue

2. Copilot Monetization:
   - Microsoft 365 Copilot: $30/user/month (400K+ enterprise customers)
   - GitHub Copilot: 1.8M paid subscribers (+40% QoQ)
   - Security Copilot: Fastest-growing enterprise product

3. Enterprise Moat:
   - Office 365 installed base: 400M+ users
   - Teams MAU: 320M (dominant collaboration platform)

Price Target: $475 | Rating: OVERWEIGHT""",
     "MSFT", "Microsoft", "segment_analysis", "2024-12-08"),

    ("DOC004", "AMD - Data Center & AI Opportunity Assessment",
     """AMD - Data Center & AI Opportunity Assessment

Competitive Positioning:

1. MI300X GPU Performance:
   - 192GB HBM3 memory (1.5x NVIDIA H100)
   - Strong inference performance for LLM workloads
   - Microsoft Azure, Oracle Cloud deployments confirmed
   - $5B+ AI GPU revenue target for 2025

2. EPYC Server CPU Dominance:
   - 33%+ server CPU market share (up from 5% in 2018)
   - Turin (Zen 5) launching H1 2025 with 192 cores

Challenges:
   - ROCm software ecosystem still lagging CUDA
   - NVIDIA mindshare advantage with AI developers

Valuation: Trading at 35x FY25E, premium for AI optionality
Rating: HOLD | PT: $165""",
     "AMD", "AMD", "investment_thesis", "2024-12-11"),

    ("DOC005", "Q1 2025 Portfolio Strategy - Technology Sector Allocation",
     """Q1 2025 Portfolio Strategy - Technology Sector Allocation

Recommended Allocation Changes:

INCREASE:
- NVDA: +3% weight (AI training demand exceeds supply)
- META: +2% weight (undervalued relative to AI investments)
- PLTR: +1% weight (government AI contracts accelerating)

MAINTAIN:
- MSFT: Current weight (balanced growth/value)
- AAPL: Current weight (services growth offsetting hardware)

REDUCE:
- AMD: -1% weight (valuation stretched vs execution risk)
- CRM: -1% weight (Agentforce adoption uncertain)

Key Themes to Monitor:
1. AI inference scaling in enterprise
2. Cloud spending reacceleration
3. China tech policy changes""",
     "PORTFOLIO", "Internal Strategy", "strategy_memo", "2025-01-05"),

    ("DOC006", "Technology Sector Risk Assessment - January 2025",
     """Technology Sector Risk Assessment - January 2025

KEY RISKS:

1. Valuation Risk (HIGH):
   - Magnificent 7 trading at 30x+ forward P/E
   - AI premium may compress if monetization disappoints

2. Regulatory Risk (MEDIUM-HIGH):
   - Google antitrust remedy could impact ad revenue
   - Apple App Store ruling may reduce services margin
   - EU Digital Markets Act enforcement increasing

3. China Exposure (MEDIUM):
   - NVDA: 20-25% revenue at risk from export controls
   - AAPL: 18% revenue, supply chain concentration

4. AI Bubble Risk (MEDIUM):
   - Infrastructure spend may front-run actual demand
   - ROI on enterprise AI investments still unproven

HEDGING RECOMMENDATIONS:
- Consider put spreads on QQQ for portfolio protection
- Maintain 5-10% cash allocation for opportunities""",
     "PORTFOLIO", "Risk Management", "risk_assessment", "2025-01-10"),
]

from pyspark.sql.types import StructType, StructField, StringType, DateType
import datetime as _dt

schema = StructType([
    StructField("doc_id", StringType()),
    StructField("title", StringType()),
    StructField("content", StringType()),
    StructField("ticker", StringType()),
    StructField("company", StringType()),
    StructField("doc_type", StringType()),
    StructField("doc_date", DateType()),
])
rows = [
    (d[0], d[1], d[2], d[3], d[4], d[5], _dt.date.fromisoformat(d[6]))
    for d in research_docs
]
(spark.createDataFrame(rows, schema)
      .write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.research_documents"))

display(spark.sql(f"SELECT doc_id, title, ticker, doc_type, doc_date FROM research_documents ORDER BY doc_id"))

## 3. Unity Catalog SQL functions — the structured "tool"

These functions are the governed, reusable interface the agent calls to answer
structured questions. Registering them in Unity Catalog means they are
discoverable, permissioned, and auditable — the agent never runs free-form SQL.
(Notebook `04` also shows how to expose the same tables through an AI/BI **Genie**
space for open-ended natural-language querying.)

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_top_holdings(max_rows INT DEFAULT 10)
RETURNS TABLE (ticker STRING, company_name STRING, total_market_value DECIMAL(18,2), total_unrealized_pnl DECIMAL(18,2))
COMMENT 'Top holdings across all portfolios ranked by total market value. Use for questions about largest positions, top holdings, biggest gainers.'
RETURN
  -- NOTE: a parameterized LIMIT is not allowed in Spark/Databricks (LIMIT must fold to
  -- a constant), and QUALIFY here rejects aggregates. So we compute a rank column in a
  -- subquery (window-over-aggregate, legal after GROUP BY) and filter it with a plain
  -- WHERE in the outer query, where the function parameter resolves fine.
  SELECT ticker, company_name, total_market_value, total_unrealized_pnl
  FROM (
    SELECT ticker,
           MAX(company_name)    AS company_name,
           SUM(market_value)    AS total_market_value,
           SUM(unrealized_pnl)  AS total_unrealized_pnl,
           ROW_NUMBER() OVER (ORDER BY SUM(market_value) DESC) AS rn
    FROM holdings
    GROUP BY ticker
  )
  WHERE rn <= max_rows;

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_portfolio_positions(portfolio STRING)
RETURNS TABLE (ticker STRING, company_name STRING, shares DECIMAL(18,4), market_value DECIMAL(18,2), unrealized_pnl DECIMAL(18,2), weight_pct DECIMAL(8,2))
COMMENT 'All positions in a given portfolio (e.g. PF001, PF002, PF003), including market value, unrealized P&L, and weight.'
RETURN
  SELECT ticker, company_name, shares, market_value, unrealized_pnl, weight_pct
  FROM holdings
  WHERE portfolio_id = get_portfolio_positions.portfolio
  ORDER BY market_value DESC;

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_ticker_exposure(symbol STRING)
RETURNS TABLE (portfolio_id STRING, shares DECIMAL(18,4), market_value DECIMAL(18,2), unrealized_pnl DECIMAL(18,2))
COMMENT 'Total exposure to a single ticker (e.g. NVDA, AAPL) across all portfolios. Use to answer how much of a stock we hold firm-wide.'
RETURN
  SELECT portfolio_id, shares, market_value, unrealized_pnl
  FROM holdings
  WHERE ticker = upper(get_ticker_exposure.symbol)
  ORDER BY market_value DESC;

In [0]:
%sql
-- Quick smoke test of the functions
SELECT * FROM get_top_holdings(5);

In [0]:
%sql
SELECT * FROM get_ticker_exposure('NVDA');

## Done

Internal structured tables, research documents, and the Unity Catalog SQL
functions are in place. Continue with **`02_vector_search`** to make the research
documents semantically searchable.